In [16]:
# Jupyter-friendly MP4 -> GIF exporter
# -----------------------------------
# Edit PARAMETERS below, then run export_mp4_to_gif()
#
# Dependencies:
#   pip install opencv-python imageio
#
# Conventions:
# - Frame indices are 0-based
# - Frame range is inclusive: [START_FRAME, END_FRAME]
# - GIF playback rate is controlled by 1 / GIF_FPS (seconds per frame)

import os
import tkinter as tk
from tkinter import filedialog, messagebox

import cv2
import imageio.v2 as imageio

from PIL import Image
import numpy as np


# PARAMETERS (edit here)

START_FRAME = 0        # inclusive
END_FRAME   = 249      # inclusive
GIF_FPS     = 50.0     # playback frame rate (frames / second)


# FILE PICKERS

def pick_input_mp4():
    root = tk.Tk()
    root.withdraw()
    root.update()

    messagebox.showinfo("Select input video", "Select an input MP4 file.")
    path = filedialog.askopenfilename(
        title="Select input MP4",
        filetypes=[("MP4 video", "*.mp4"), ("All files", "*.*")]
    )

    root.destroy()
    return path if path else None


def pick_output_gif(default_name="output.gif"):
    root = tk.Tk()
    root.withdraw()
    root.update()

    messagebox.showinfo("Save output GIF", "Choose where to save the output GIF.")
    path = filedialog.asksaveasfilename(
        title="Save GIF as",
        defaultextension=".gif",
        initialfile=default_name,
        filetypes=[("GIF", "*.gif"), ("All files", "*.*")]
    )

    root.destroy()
    return path if path else None


# VIDEO / GIF UTILITIES

def get_total_frames(video_path):
    cap = cv2.VideoCapture(video_path)
    if not cap.isOpened():
        cap.release()
        raise RuntimeError(f"Could not open video: {video_path}")
    total = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    cap.release()
    if total <= 0:
        raise RuntimeError("Could not determine total frame count.")
    return total


def read_frames_rgb(video_path, start_frame, end_frame):
    cap = cv2.VideoCapture(video_path)
    if not cap.isOpened():
        cap.release()
        raise RuntimeError(f"Could not open video: {video_path}")

    cap.set(cv2.CAP_PROP_POS_FRAMES, start_frame)

    frames = []
    current = start_frame
    while current <= end_frame:
        ok, frame_bgr = cap.read()
        if not ok:
            break
        frames.append(cv2.cvtColor(frame_bgr, cv2.COLOR_BGR2RGB))
        current += 1

    cap.release()

    if not frames:
        raise RuntimeError("No frames read.")
    return frames


def save_gif(frames_rgb, output_gif_path, gif_fps):
    duration_ms = int(round(1000.0 / gif_fps))  
    pil_frames = [Image.fromarray(np.asarray(f, dtype=np.uint8)) for f in frames_rgb]  # <<< CHANGE
    pil_frames[0].save(  
        output_gif_path,
        save_all=True,
        append_images=pil_frames[1:],
        duration=duration_ms,
        loop=0,          # 0 = loop forever
    )


# MAIN WORKFLOW

def export_mp4_to_gif():
    mp4_path = pick_input_mp4()
    if not mp4_path:
        return

    base = os.path.splitext(os.path.basename(mp4_path))[0]
    gif_path = pick_output_gif(default_name=f"{base}.gif")
    if not gif_path:
        return

    total_frames = get_total_frames(mp4_path)

    if not (0 <= START_FRAME <= END_FRAME < total_frames):
        raise ValueError(
            f"Invalid frame range [{START_FRAME}, {END_FRAME}] "
            f"for total_frames={total_frames}."
        )

    frames_rgb = read_frames_rgb(mp4_path, START_FRAME, END_FRAME)
    save_gif(frames_rgb, gif_path, GIF_FPS)

    root = tk.Tk()
    root.withdraw()
    root.update()
    messagebox.showinfo("Done", f"Saved GIF:\n{gif_path}")
    root.destroy()



# Run from notebook cell:

export_mp4_to_gif()
